# Choosing how many epochs to train the base model

### Imports

In [1]:
import os
import json

In [2]:
%ls

README.md                            pretraining_models.ipynb
__pycache__/                         results/
_old/                                trainer/
data/                                unlearn/
evaluation/                          visualize_pretraining_results.ipynb
master_experiment.ipynb              visualize_results.ipynb
models/                              wandb/


In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# from trainer.utils import training_regimen_lr_annealing


### Set configs for the pretraining

In [4]:
device = "mps" if torch.mps.is_available() else "cpu"
pretraining_config = {

    "description": "Pre-training ResNet on SVHN",
    
    "device": device,
    "model_class": "ResNet",
    "data": {
        "dataset": "SVHN",
        "batch_size": 1024,
        "num_workers": 0,
        },

    "training": {
        "num_epochs": [1, 5, 10, 15, 20, 30],
        "num_runs": 3,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "batch_print_freq": 5,
        },
}

### Protocol for several runs

In [5]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/jerrymoncus/.netrc.
wandb: Currently logged in as: jjmoncus (jjmoncus706) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import os
import json
import random
import glob
import torch
import wandb
import torch.nn as nn
import torch.optim as optim
from models.archs.utils import init_model
from data.dataloaders import load_dataloaders_for_experiment
from trainer.utils import init_folder_if_not_exists, training_regimen_lr_annealing
from trainer.val import validate
from models.archs.utils import init_model

def run_pretraining(config):

    print("-"*75)
    print("-"*13 + "  " + f"EVALUATING # OF EPOCHS FOR TRAINING {config['model_class']}" + "  " + "-"*13)
    print("-"*75 + "\n")

    # init wandb
    wandb.init(
      project="Verifying-Unlearning-2026",
      name=f"Pretraining Experiments - {config['model_class']}",
      config=config,
      reinit=True
    )

    # Make experiment results folder if it doesnt already exist
    results_folder = init_folder_if_not_exists( f"results/pretraining/seed_{config['GRAND_SEED']}" )
    
    # Save the config for this experiment to the main results folder
    with open(os.path.join(results_folder, "experiment_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # make model checkpoints folder for this seed if it doesn't exist yet
    checkpoints_folder = init_folder_if_not_exists( f"models/model_checkpoints/pretrained/seed_{config['GRAND_SEED']}" )


    for num_epochs in config["training"]["num_epochs"]:

        epoch_results_folder = init_folder_if_not_exists( os.path.join(results_folder, f"{config["data"]["dataset"]}_{config['model_class']}_{num_epochs}_epochs") )
        epoch_checkpoints_folder = init_folder_if_not_exists( os.path.join(checkpoints_folder, f"{config["data"]["dataset"]}_{config['model_class']}_{num_epochs}_epochs") )


        # get some data (does not change in between runs)
        train_loader, _, test_loader = load_dataloaders_for_experiment(
            name = config["data"]["dataset"],
            batch_size=config["data"]["batch_size"], 
            num_workers=config["data"]["num_workers"], 
            seed=config["GRAND_SEED"], 
            class_to_replace=None, 
            percent_to_replace=None,
            val=False
            )

        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #
        # ---------------- TRAIN A BASE MODEL, FROM WHICH UNLEARNING BEGINS ----------------- #
        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #


        print("-"*55)
        print("-"*13 + "  " + f"TESTING: {num_epochs} EPOCHS" + "  " + "-"*13)
        print("-"*55 + "\n")

        for i in range(1, config["training"]["num_runs"] + 1):

            # seed PyTorch, numpy, and Python RNG for this run so initializations
            # are reproducible and intentionally distinct across runs
            run_seed = config["GRAND_SEED"] * 100 + i
            torch.manual_seed(run_seed)
            torch.cuda.manual_seed_all(run_seed)
            import numpy as np
            np.random.seed(run_seed)
            random.seed(run_seed)

            # init model, opt, criterion, and scheduler
            empty_model = init_model(model_class = config["model_class"]).to(config["device"])
            criterion = nn.CrossEntropyLoss()
            opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                opt, 
                T_max=num_epochs, 
                eta_min=1e-6
            )

            # train (wandb logging underneath, dont need to re-log training accuracy)
            base_model_path = os.path.join(epoch_checkpoints_folder, f"{config['model_class']}_{i}.pth")
            trained_model, opt, scheduler, _, _, _, _ = training_regimen_lr_annealing(
                empty_model, 
                train_loader,
                opt, 
                criterion, 
                scheduler, 
                device = config["device"], 
                num_epochs=num_epochs, 
                model_path = base_model_path,
                print_freq = config["training"]["batch_print_freq"])

            # evaluate trained model on some test
            print(f"Evaluating model trained for {num_epochs} epochs on test set...\n") 
            _, train_acc, _, _, _, _, _ = validate(
                train_loader, 
                trained_model, 
                criterion, 
                print_freq = config["training"]["batch_print_freq"],
                device = config["device"]
            )
            
            _, test_acc, _, _, _, _, _ = validate(
                test_loader, 
                trained_model, 
                criterion, 
                print_freq = config["training"]["batch_print_freq"],
                device = config["device"]
            )

            print(f"Train accuracy: {train_acc:.4f}\n")
            print(f"Test accuracy: {test_acc:.4f}\n")
            
            results = {
                "num_epochs": num_epochs,
                "run_seed": run_seed,
                "train_acc": train_acc,
                "test_acc": test_acc
                }

            # save results
            with open(os.path.join(epoch_results_folder, f"results_{i}.json"), "w") as f:
                json.dump(results, f, indent=4)

            # Save checkpoint
            print(f"Saving base model to {base_model_path}...")
            torch.save(trained_model.state_dict(), base_model_path)
                    
    wandb.finish()

    print("-"*70)
    print("-"*19 + "  " + f"FINISHED PRETRAINING" + "  " + "-"*19)
    print("-"*70 + "\n")


### Check metrics on unlearned models

In [7]:
# MAKE A RANDOM SEED
pretraining_config["GRAND_SEED"] = 3
# DO EXP
run_pretraining(config = pretraining_config)

---------------------------------------------------------------------------
-------------  EVALUATING # OF EPOCHS FOR TRAINING ResNet  -------------
---------------------------------------------------------------------------



wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


========== DATALOADER INFO
Dataset: SVHN
Train: 73257 images for training
Test: 26032 images for testing
Training augmentation = None
Validation/Test augmentation = None


-------------------------------------------------------
-------------  TESTING: 1 EPOCHS  -------------
-------------------------------------------------------

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][4/72]	Loss 3.0472 (4.8954)	Accuracy 15.625 (11.074)	Entropy 4.5429 (6.0911)	M-Entropy 3.0041 (4.8551)	Time 8.99
Epoch: [1][9/72]	Loss 2.0464 (3.5609)	Accuracy 27.637 (17.080)	Entropy 2.2147 (4.4114)	M-Entropy 1.9022 (3.4702)	Time 7.86
Epoch: [1][14/72]	Loss 1.8037 (3.0045)	Accuracy 37.207 (22.109)	Entropy 1.8549 (3.5919)	M-Entropy 1.6977 (2.9059)	Time 7.98
Epoch: [1][19/72]	Loss 1.3455 (2.6390)	Accuracy 58.301 (29.058)	Entropy 1.6380 (3.1319)	M-Entropy 1.1995 (2.5303)	Time 8.10
Epoch: [1][24/72]	Loss 0.9791 (2.3352)	Accuracy 69.824 (36.070)	Entropy 1.1514 (2.7720)	M-Entropy 0.91

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x284ea1a60>> (for post_run_cell), with arguments args (<ExecutionResult object at 285d9bc80, execution_count=7 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 285d9bc20, raw_cell="# MAKE A RANDOM SEED
pretraining_config["GRAND_SEE.." transformed_cell="# MAKE A RANDOM SEED
pretraining_config["GRAND_SEE.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/jerrymoncus/data_science_projects/UCL/thesis/verifying_unlearning_2026/pretraining_models.ipynb#X14sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost